# Martinez-Conde et al. (2019) on CRCNS pvc-12

Paper: Martinez-Conde S, McCamy MB, Troncoso XG, Otero-Millan J, Macknik SL (2019). Area V1
responses to illusory corner-folds in Vasarely's nested squares and the Alternating Brightness Star
illusions. PLoS ONE 14(3): e0210941, doi:10.1371/journal.pone.0210941.

Claims: `claims/martinez-conde-2019.yaml`, written from the version of record (PMC6438452, Europe PMC
full text: main text, Methods, legends, Table 1 and the legends of the supporting information; S1 and
S2 Table themselves are not in the text).

Dataset: CRCNS pvc-12, doi:10.6080/K05D8Q1T, the single zip file `pvc-12/live/V1_Vasarely_data.zip`
(MD5 `e1301fe463e72332e2303b2053ee0559` in the data set's `checksums.md5`), streamed from the public S3
bucket `crcnsarchive`. Version: CRCNS data sets are not versioned, so the version is recorded as
`unversioned` and the MD5 sum above identifies the file read; there is no other version to choose
from, and the data description (version 0.7, 15 Feb 2018) names this paper. Checking the MD5 would mean reading
the whole 175 MB file, so the notebook confirms that `checksums.md5` still lists this sum.

What the dataset holds: per neuron and condition a spike rate map (spikes per eye sample at each
display unit, smoothed, and an aligned copy in the region of interest, "Standard") and the stimulus.
It holds no spike times and no eye positions.

| Claim | Tested here |
|---|---|
| neuron-counts | tested |
| spikes-per-condition | blocked, data missing (no spike counts; eye time in 1 of 75 files) |
| s4-baseline-no-difference | tested |
| fig4a-above-baseline | tested |
| fig4b-spike-rate-difference | tested |
| fig4b-pairwise-significance | tested |
| fig5-average-maps | tested |
| table1-edge, table1-p180, table1-s90, table1-s52, table1-s40 | tested |
| fig6-midpoint-representative | tested |
| fig7a-rms-contrast-fit | tested |
| fig7c-fold-edge-equivalence | tested |
| fig7c-fold-edge-significance | tested |
| fig7d-nonlinear | tested |
| fig7d-dog-similar | blocked, data missing (the DOG output of reference 3 is not public) |
| s3-latency-peak | blocked, data missing (no spike times or eye positions) |

Star claims rest on the 12 Star neurons whose files carry the aligned frame; the other 15 Star files
cannot be registered to the stimulus (data check, below).

In [ ]:
import hashlib, io, resource, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit, minimize
from statsmodels.stats.multitest import multipletests
import fsspec
import vasarely as v

T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dataset", v.DATASET, v.VERSION, "doi", v.DOI)
checksums = fsspec.open(v.ZIP_URL.rsplit("/", 1)[0] + "/checksums.md5", "rt").open().read()
print(checksums)
assert v.ZIP_MD5 in checksums

## Data check

Every file is listed, and every map file is read (only the aligned maps and polygons are kept). The
cell prints the counts by condition and monkey, the fields each file holds, the stimulus type of each
file, and the checks of the description against the values.

In [ ]:
z = v.open_zip()
files = v.file_table(z)
recs = {(r.folder, r.code): v.load_map(z, r.folder, r.code) for r in files.itertuples()}
files["stimulus"] = [v.stimulus_type(z, r.folder, r.code) for r in files.itertuples()]
files["standard"] = [recs[(r.folder, r.code)].get("standard_fields") is not None for r in files.itertuples()]
files["base_stored"] = ["base_stored" in recs[(r.folder, r.code)] for r in files.itertuples()]
files["frame"] = [str(recs[(r.folder, r.code)]["smoothed"].shape) if "smoothed" in recs[(r.folder, r.code)] else "-"
                  for r in files.itertuples()]
print("map files by folder:", files.groupby("folder").size().to_dict())
print("unique neurons with any file:", files.neuron.nunique(),
      "by monkey:", files.drop_duplicates("neuron").monkey.value_counts().to_dict())
print("\nstimulus type by folder and prefix:")
print(files.groupby(["folder", "prefix", "stimulus"]).size().to_string())
print("\naligned frame (Standard), stored BaseActivity and frame size by folder:")
print(files.groupby(["folder", "standard", "base_stored", "frame"]).size().to_string())
print("\nfiles whose map file is not MATLAB 5 (not read):", [r.code for r in files.itertuples()
                                                          if recs[(r.folder, r.code)]["fields"] is None])
print("peak memory so far, MB:", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024)

In [ ]:
# The data set's own list of cells (V1_Vasarely_data_SMC.xlsx) against the files.
sheet = v.read_sheet(z)
print("cells listed:", len(sheet), " with no condition marked:", list(sheet.neuron[~sheet[["Corners", "Star", "Edge-compare"]].any(axis=1)]))
print("marked per condition:", {c: int(sheet[c].sum()) for c in ["Corners", "Star", "Edge-compare"]})
print("cells listed by monkey:", sheet.neuron.str.extract(r"([JA])")[0].value_counts().to_dict())
mism = []
for c in ["Corners", "Star", "Edge-compare"]:
    in_files = set(files.neuron[files.folder == c])
    in_sheet = set(sheet.neuron[sheet[c]])
    if in_files != in_sheet:
        mism.append((c, sorted(in_files - in_sheet), sorted(in_sheet - in_files)))
print("sheet and files disagree on (condition, only in files, only in sheet):", mism or "nothing")

In [ ]:
# Units. `image` is described as raw spike counts. Most of its nonzero values are exactly 1/n (one
# spike over n eye samples), and the one file that also stores eye time (ABC050900J1A, field
# `eyetime`, integer 1 ms samples) gives integer counts when the two are multiplied, so `image` is
# spikes per eye sample, i.e. spikes per ms; x 1000 gives spikes/s.
r = recs[("Corners", "ABC050900J1A")]
counts = r["screen_image"] * r["screen_eyetime"]
print("eyetime values (first):", np.unique(r["screen_eyetime"])[:6], " max |count - round(count)|:",
      float(np.abs(counts - np.round(counts)).max()), " total spikes in this file:", int(np.round(counts).sum()))
share = [r["screen_one_over_n"] for r in recs.values() if r.get("fields") and "screen_one_over_n" in r]
print("share of nonzero screen-frame `image` values equal to 1/n, over %d files: min %.4f, median %.4f"
      % (len(share), min(share), np.median(share)))
# Rates in spikes/s (x 1000) and their ranges over all aligned maps
sm = [r["smoothed"] for r in recs.values() if "smoothed" in r]
print("smoothed spike rate maps, spikes/s: min %.2f, median of per-map max %.1f, max %.1f; share of zero pixels %.3f"
      % (min(m.min() for m in sm), np.median([m.max() for m in sm]), max(m.max() for m in sm),
         np.mean([np.mean(m == 0) for m in sm])))
# smoothed_image_norm = (smoothed - BaseActivity) / (max - BaseActivity)
errs = []
for r in recs.values():
    if "norm" in r and "base_stored" in r:
        s, b = r["smoothed"], r["base_stored"]
        errs.append(float(np.abs((s - b) / (s.max() - b) - r["norm"]).max()))
print("smoothed_image_norm vs (smoothed - BaseActivity)/(max - BaseActivity): files", len(errs),
      "; files where it holds to 1e-6:", int(np.sum(np.array(errs) < 1e-6)), "; largest differences in the others:",
      np.round(sorted(e for e in errs if e >= 1e-6), 2))

In [ ]:
# Geometry: the fold lines from the polygons are the same in every aligned frame of a condition
# (the 6 Corners frames of 257 x 193 are the 273 x 209 frame cut by 8 DU on each side).
geo = []
for (f, c), r in recs.items():
    if "smoothed" in r:
        g = v.geometry(r)
        for fold in v.FOLDS[f]:
            for x, y0, y1 in g[fold]:
                geo.append(dict(folder=f, frame=str(r["smoothed"].shape), fold=fold, x=round(x, 1),
                                y0=round(y0, 1), y100=round(y1, 1), length_10pct=round(abs(y1 - y0) / 10, 2)))
geo = pd.DataFrame(geo).drop_duplicates()
print(geo.to_string(index=False))

In [ ]:
# Selection. Star: 2 of the 29 files in the Star folder (prefix ABS) hold a Corners-type stimulus
# (a single notched corner), not the twelve-pointed Star, and have no aligned frame; 15 of the
# remaining 27 Star files have no aligned frame either. The screen-frame maps are in gaze
# coordinates: measured at the stimulus's screen polygons along the aligned diagonal they do not
# match the aligned frame, so the receptive-field offset that the aligned frame applies is not in
# the files and those 15 cannot be registered to the stimulus.
cmp = []
for (f, c), r in recs.items():
    if f == "Star" and "smoothed" in r:
        ang = v.star_aligned_angle(r)
        scr = v.star_screen_measures(r, ang)
        cmp.append(dict(code=c, aligned_angle=round(ang, 1), midpoint_aligned_frame=v.measures(r)["midpoint_S52"],
                        midpoint_screen_frame=scr["midpoint_S52"]))
cmp = pd.DataFrame(cmp)
print(cmp.round(2).to_string(index=False))
print("Pearson r between the two: %.2f; median ratio screen/aligned %.2f" %
      (np.corrcoef(cmp.midpoint_aligned_frame, cmp.midpoint_screen_frame)[0, 1],
       np.median(cmp.midpoint_screen_frame / cmp.midpoint_aligned_frame)))
print("aligned diagonals (screen degrees) of the 12:", sorted(cmp.aligned_angle.unique()),
      "; the star is drawn with the same screen orientation in every file, so the diagonal is not in the stimulus")

In [ ]:
# The analysed set: every file with an aligned frame and its condition's stimulus.
use = files[files.standard & ~files.stimulus.str.startswith("v7.3")].copy()
rows = []
for r in use.itertuples():
    m = v.measures(recs[(r.folder, r.code)])
    mu = v.measures(recs[(r.folder, r.code)], smoothed=False)
    m.update(folder=r.folder, code=r.code, neuron=r.neuron, monkey=r.monkey,
             **{f"raw_{k}": val for k, val in mu.items() if k.startswith(("midpoint_", "base_main"))})
    rows.append(m)
per = pd.DataFrame(rows)
COND = {"Edge-compare": "Square and Edge", "Star": "Star", "Corners": "Corners"}
print("analysed neurons per condition:", per.groupby("folder").size().rename(COND).to_dict())
print("by monkey:", per.groupby(["folder", "monkey"]).size().to_dict())
print("dropped: Star files without aligned frame %d, misfiled Corners-type files in Star %d"
      % (int(((files.folder == "Star") & ~files.standard & (files.prefix == "ABD")).sum()),
         int((files.prefix == "ABS").sum())))
# Spatial registration check (the maps carry no time, so the analogue of decoding with shifted labels
# is moving the midpoint square 32 DU sideways, off the fold): fold minus off-fold, per neuron
for f, folds in v.FOLDS.items():
    d = per[per.folder == f]
    for fold in folds:
        diff = d[f"midpoint_{fold}"] - d[f"offfold_{fold}"]
        w = stats.wilcoxon(diff)
        print(f"{fold:8s} n={len(d):2d} on fold %.2f, 32 DU off fold %.2f spikes/s, fold higher in %d, Wilcoxon p = %.2g"
              % (d[f"midpoint_{fold}"].mean(), d[f"offfold_{fold}"].mean(), (diff > 0).sum(), w.pvalue))
print("load and measures, s: %.0f" % (time.time() - T0))

Baselines. The claims file defines the absolute baseline as the mean of the spike rate map in four
(Square and Edge, Star) or two (Corners) 16 x 16 DU squares in 0% luminance. For Square and Edge and
Star the main placement is inside the corners of the circumscribing square; centred on the corners is a
variant. The Corners squares are only drawn in Fig 2C, which is not in the text; the main placement is
the two bottom corners of the aligned frame, an assumption. The stored `BaseActivity` (38 of 60
analysed files) is a further variant. The next cell compares them.

In [ ]:
b = per[["folder", "code", "base_main", "base_centred", "base_stored"]].copy()
for f in v.FOLDS:
    d = b[(b.folder == f) & b.base_stored.notna()]
    print(f"{COND[f]:16s} n with stored = {len(d):2d}; median main %.2f, centred %s, stored %.2f spikes/s; "
          "Spearman main vs stored %.2f" % (d.base_main.median(),
          "%.2f" % d.base_centred.median() if d.base_centred.notna().any() else "-",
          d.base_stored.median(), stats.spearmanr(d.base_main, d.base_stored)[0] if len(d) > 2 else np.nan))
print("stored BaseActivity equal to 0 in:", list(b.code[b.base_stored == 0]))

## neuron-counts

> Of 118 V1 neurons recorded from two monkeys (102 from monkey J, 16 from monkey A), 60 were
> discarded for technical problems and 58 analysed; 7 were tested in all three conditions, 11 in two
> and 40 in one, giving 25 neurons for Square and Edge, 30 for Star and 28 for Corners.

Neurons are identified by their code (the cell name, with the file prefix and any trailing session
letter removed), as in the data set's sheet.

In [ ]:
def overlap(tab):
    n = tab.groupby("neuron").folder.nunique()
    return dict(neurons=int(n.size), all_three=int((n == 3).sum()), two=int((n == 2).sum()), one=int((n == 1).sum()))

expected = dict(neurons=58, **{"Square and Edge": 25, "Star": 30, "Corners": 28}, all_three=7, two=11, one=40)
readings = {
    "files, every folder (Star incl. the 2 misfiled)": files,
    "files, condition stimulus only": files[files.prefix != "ABS"],
    "analysed here (aligned frame)": use,
}
rows = []
for name, tab in readings.items():
    row = dict(reading=name, **overlap(tab), **tab.groupby("folder").neuron.nunique().rename(COND).to_dict())
    rows.append(row)
sheet_long = sheet.melt(id_vars="neuron", var_name="folder", value_name="x")
sheet_long = sheet_long[sheet_long.x]
rows.append(dict(reading="data set sheet (xlsx)", **overlap(sheet_long),
                 **sheet_long.groupby("folder").neuron.nunique().rename(COND).to_dict()))
rows.append(dict(reading="paper", **expected))
counts = pd.DataFrame(rows)[["reading", "neurons", "Square and Edge", "Star", "Corners", "all_three", "two", "one"]]
print(counts.to_string(index=False))
print("monkeys in the files:", files.drop_duplicates("neuron").monkey.value_counts().to_dict(),
      "; the sheet lists", len(sheet), "cells, of which", int((~sheet[["Corners", "Star", "Edge-compare"]].any(axis=1)).sum()),
      "have no condition")

**Result: not reproduced.** The files hold 53 neurons (49 from monkey J, 4 from A) where the paper analysed 58: 22 for Square and Edge (paper 25), 29 in the Star folder, of which 27 with the Star stimulus (paper 30), and 26 for Corners (paper 28). 7 neurons have all three conditions, as in the paper, but 10 have two and 36 one (paper 11 and 40); without the 2 misfiled Star files, 8 and 38. The data set's sheet lists 58 cells, as many as the paper analysed, but 5 of them have no condition marked and no files, and its marks agree with the files. Only 12 of the 27 Star neurons can be analysed (data check).

## spikes-per-condition

> Each experimental condition required at least 10,000 spikes, and the mean number of spikes per
> condition was 35,998 (STD 25,981).

Blocked: the maps are spike counts divided by eye time, and eye time is stored in one file of 75
(ABC050900J1A, 15,644 spikes, printed in the data check). Neither the spike count maps nor the spike
times are in the dataset.

## s4-baseline-no-difference

> Absolute baseline activity did not differ significantly across the three visual stimuli (Square and
> Edge, Star, Corners).

Readings of the claims file, each at alpha 0.05: (a) pairwise Wilcoxon rank sum tests, Benjamini and
Hochberg; (b) Kruskal-Wallis; (c) one-way ANOVA. Variants: the centred placement and the stored
`BaseActivity`.

In [ ]:
def baseline_tests(col, data):
    groups = {COND[f]: data.loc[data.folder == f, col].dropna().values for f in v.FOLDS}
    pairs = [("Square and Edge", "Star"), ("Square and Edge", "Corners"), ("Star", "Corners")]
    p_raw = [stats.ranksums(groups[a], groups[b]).pvalue for a, b in pairs]
    p_bh = multipletests(p_raw, method="fdr_bh")[1]
    out = {f"(a) {a} vs {b}": f"p = {p:.3f}, BH {q:.3f}" for (a, b), p, q in zip(pairs, p_raw, p_bh)}
    out["(b) Kruskal-Wallis"] = "p = %.3f" % stats.kruskal(*groups.values()).pvalue
    out["(c) one-way ANOVA"] = "p = %.3f" % stats.f_oneway(*groups.values()).pvalue
    desc = {k: "n=%d median %.2f (IQR %.2f to %.2f, range %.2f to %.2f)" % (len(g), np.median(g), *np.percentile(g, [25, 75]), g.min(), g.max())
            for k, g in groups.items()}
    return out, desc

data = per.copy()
data.loc[data.folder == "Corners", "base_centred"] = data.loc[data.folder == "Corners", "base_main"]
for label, col in [("main placement", "base_main"), ("variant, centred squares (Corners as main)", "base_centred"),
                   ("variant, stored BaseActivity", "base_stored")]:
    tests, desc = baseline_tests(col, data)
    print(label)
    for k, val in desc.items():
        print("   ", k, val, "spikes/s")
    for k, val in tests.items():
        print("   ", k, val)

**Result: not reproduced (resting on the assumed baseline placement).** With the baseline squares placed as the claims file describes (inside the corners of the circumscribing square; the bottom corners of the frame for Corners, an assumption), all three readings find a difference: Square and Edge (median 7.68 spikes/s, n = 22) is higher than Star (1.89, n = 12; BH p = 0.016) and Corners (2.45, n = 26; BH p = 0.008), Kruskal-Wallis p = 0.004, ANOVA p = 0.001. Centring the squares on the corners gives the same (Kruskal-Wallis p = 0.007). With the stored `BaseActivity`, on the 38 files that have it, no test finds a difference (Kruskal-Wallis p = 0.114, smallest BH p = 0.109), with medians 5.64 (Square and Edge), 2.68 (Star) and 1.32 spikes/s (Corners).

## fig4a-above-baseline

> For every fold (P180, S90, S52, S40), most neurons' midpoint fold responses lie above their absolute
> baseline activity, and the mass center of each fold's points lies above the line of unit slope, with
> the mass center's distance from that line increasing as the fold angle decreases.

Readings for the mass centre: (a) mean of raw values; (b) mean of log10 values (pairs with a zero
response or baseline are left out of (b), and counted).

In [ ]:
FOLDS4 = [("P180", "Edge-compare"), ("S90", "Edge-compare"), ("S52", "Star"), ("S40", "Corners")]
ANGLE = {"P180": 180, "S90": 90, "S52": 52, "S40": 40}
rows = []
for fold, f in FOLDS4:
    d = per[per.folder == f]
    resp, base = d[f"midpoint_{fold}"].values, d.base_main.values
    ok = (resp > 0) & (base > 0)
    rows.append(dict(fold=fold, n=len(d), above=int((resp > base).sum()), share_above=(resp > base).mean(),
                     centre_base=base.mean(), centre_resp=resp.mean(), distance_a=resp.mean() - base.mean(),
                     n_log=int(ok.sum()), distance_b_log10=np.log10(resp[ok]).mean() - np.log10(base[ok]).mean()))
fa = pd.DataFrame(rows)
print(fa.round(3).to_string(index=False))
for col in ["distance_a", "distance_b_log10"]:
    print(col, "ordered P180 < S90 < S52 < S40:", bool(np.all(np.diff(fa[col].values) > 0)))
# by monkey (A has few neurons)
print(per.assign(**{"d_" + fold: per[f"midpoint_{fold}"] - per.base_main for fold, _ in FOLDS4})
      .groupby(["folder", "monkey"])[[ "d_" + fold for fold, _ in FOLDS4]].mean().round(2).to_string())
fig, ax = plt.subplots(1, 4, figsize=(14, 3.4))
for a, (fold, f) in zip(ax, FOLDS4):
    d = per[per.folder == f]
    a.loglog(d.base_main.clip(lower=0.1), d[f"midpoint_{fold}"].clip(lower=0.1), "o", ms=4)
    a.plot([0.1, 100], [0.1, 100], "k:")
    a.set_title(fold); a.set_xlabel("baseline, spikes/s")
ax[0].set_ylabel("midpoint response, spikes/s")
plt.tight_layout(); plt.show()
print("values below 0.1 spikes/s drawn at 0.1:", int(sum((per[f"midpoint_{fold}"][per.folder == f] < 0.1).sum() + (per.base_main[per.folder == f] < 0.1).sum() for fold, f in FOLDS4)))

**Result: reproduced.** Most neurons lie above the unit line for every fold: 20 of 22 for P180 and for S90, 12 of 12 for S52 and 26 of 26 for S40. Every mass centre is above the line, and its distance from it increases as the angle decreases under both readings: raw means 3.88, 8.19, 19.01 and 26.48 spikes/s for P180, S90, S52 and S40, log10 means 0.18, 0.30, 0.91 and 1.04 (2 S40 neurons with a value of 0 are left out of the log reading). S52 rests on 12 of the paper's 30 Star neurons.

## fig4b-spike-rate-difference

> The mean spike rate difference at the fold midpoint decreases with increasing fold angle, from S40
> (25.60 spikes/s) through S52 (18.10) and S90 (10.69) to P180 (4.87).

Spike rate difference = midpoint response minus absolute baseline, per neuron; mean and SEM over
neurons. Variants: the unsmoothed map, the centred baseline squares, the stored `BaseActivity`.

In [ ]:
EXPECTED = {"S40": (25.60, 4.18), "S52": (18.10, 2.39), "S90": (10.69, 1.94), "P180": (4.87, 1.15)}
for fold, f in FOLDS4:
    per.loc[per.folder == f, f"diff_{fold}"] = per[f"midpoint_{fold}"] - per.base_main
    per.loc[per.folder == f, f"diffraw_{fold}"] = per[f"raw_midpoint_{fold}"] - per.raw_base_main
    per.loc[per.folder == f, f"diffc_{fold}"] = per[f"midpoint_{fold}"] - (per.base_centred if f != "Corners" else per.base_main)
    per.loc[per.folder == f, f"diffs_{fold}"] = per[f"midpoint_{fold}"] - per.base_stored

def summary(prefix):
    rows = []
    for fold, f in reversed(FOLDS4):
        x = per.loc[per.folder == f, f"{prefix}_{fold}"].dropna()
        e, tol = EXPECTED[fold]
        rows.append(dict(fold=fold, n=len(x), mean=x.mean(), sem=x.sem(), expected=e, tolerance=tol,
                         within=abs(x.mean() - e) <= tol))
    t = pd.DataFrame(rows)
    return t, bool(np.all(np.diff(t["mean"].values) < 0))

for label, prefix in [("main (smoothed map, baseline inside corners)", "diff"), ("variant, unsmoothed map", "diffraw"),
                      ("variant, centred baseline squares", "diffc"), ("variant, stored BaseActivity (subset)", "diffs")]:
    t, ordered = summary(prefix)
    print(label, "| ordered S40 > S52 > S90 > P180:", ordered)
    print(t.round(2).to_string(index=False))
print("by monkey (main):")
print(per.groupby(["folder", "monkey"])[[f"diff_{fold}" for fold, _ in FOLDS4]].agg(["mean", "count"]).round(2).dropna(how="all", axis=1).to_string())

**Result: partially reproduced.** The means are ordered S40 > S52 > S90 > P180: 26.48 ± 4.28 (n = 26), 19.01 ± 3.00 (n = 12), 8.19 ± 1.98 (n = 22) and 3.88 ± 0.91 spikes/s (n = 22). S40, S52 and P180 are within the paper's SEM. S90 is 2.50 below 10.69, just outside its tolerance of 1.94. The order holds under every variant. Under each variant one or two values fall outside tolerance: S90 with centred baseline squares; S52 and P180 with the unsmoothed map or the stored `BaseActivity`. S52 rests on 12 of 30 Star neurons.

## fig4b-pairwise-significance

> The midpoint spike rate difference for S40 is significantly larger than for S90 and for P180, and
> that for S52 is significantly larger than for P180.

Wilcoxon rank sum tests, Benjamini and Hochberg; families (a) all six pairs, (b) the three named.

In [ ]:
vals = {fold: per.loc[per.folder == f, f"diff_{fold}"].dropna().values for fold, f in FOLDS4}
pairs = [("S40", "S52"), ("S40", "S90"), ("S40", "P180"), ("S52", "S90"), ("S52", "P180"), ("S90", "P180")]
named = [("S40", "S90"), ("S40", "P180"), ("S52", "P180")]
p = {pr: stats.ranksums(vals[pr[0]], vals[pr[1]]).pvalue for pr in pairs}
bh6 = dict(zip(pairs, multipletests([p[pr] for pr in pairs], method="fdr_bh")[1]))
bh3 = dict(zip(named, multipletests([p[pr] for pr in named], method="fdr_bh")[1]))
rows = [dict(pair=f"{a} > {b}", median_a=np.median(vals[a]), median_b=np.median(vals[b]),
             mean_difference=vals[a].mean() - vals[b].mean(), p=p[(a, b)], bh_six=bh6[(a, b)],
             bh_three=bh3.get((a, b), np.nan), named=(a, b) in named) for a, b in pairs]
pw = pd.DataFrame(rows)
print(pw.round(4).to_string(index=False))
for fam in ["bh_six", "bh_three"]:
    sig = pw[pw.named]
    print(fam, ": named pairs significant in the stated direction:",
          int(((sig[fam] < 0.05) & (sig.mean_difference > 0)).sum()), "of 3")

**Result: reproduced.** S40 is larger than S90 (BH p = 0.0001 over six pairs) and than P180 (p < 0.0001), and S52 is larger than P180 (p < 0.0001); all three are significant in the stated direction over six pairs and over the three named. Of the other pairs, S52 is larger than S90 (BH p = 0.0047), and neither S40 against S52 (0.31) nor S90 against P180 (0.076) is significant.

## fig5-average-maps

> The population-average spike rate difference maps show neural activity increasing as the corner
> angle decreases.

Average over neurons of the spike rate difference maps (aligned map minus baseline) per condition.
The 6 Corners frames of 257 x 193 are placed in the 273 x 209 frame (8 DU margin each side) and the
average is taken where data exist. Reading: the average map in the 8 x 8 DU midpoint squares; variant:
the mean of the average map over the ten blocks.

In [ ]:
avg = {}
for f in v.FOLDS:
    maps, ref = [], None
    for r in per[per.folder == f].itertuples():
        rec = recs[(f, r.code)]
        m = rec["smoothed"] - r.base_main
        if f == "Corners" and m.shape == (257, 193):
            big = np.full((273, 209), np.nan); big[8:-8, 8:-8] = m; m = big
        else:
            ref = rec
        maps.append(m)
    avg[f] = (np.nanmean(np.stack(maps), axis=0), ref)
rows = []
for fold, f in FOLDS4:
    m, ref = avg[f]
    g = v.geometry(ref)
    rows.append(dict(fold=fold, midpoint_of_average_map=v.midpoint(m, g[fold]),
                     mean_of_blocks_of_average_map=float(np.mean([v.blocks(m, fl) for fl in g[fold]]))))
f5 = pd.DataFrame(rows)
print(f5.round(2).to_string(index=False))
for col in f5.columns[1:]:
    print(col, "ordered S40 > S52 > S90 > P180:", bool(np.all(np.diff(f5[col].values) > 0)))
fig, ax = plt.subplots(1, 3, figsize=(12, 4.5))
for a, f in zip(ax, v.FOLDS):
    m = avg[f][0]
    im = a.imshow(m, cmap="magma", vmin=0, vmax=40); a.set_title(COND[f])
    print(COND[f], "average map range %.1f to %.1f spikes/s; pixels above 40: %d, below 0: %d"
          % (np.nanmin(m), np.nanmax(m), int(np.sum(m > 40)), int(np.sum(m < 0))))
plt.colorbar(im, ax=ax, label="spike rate difference, spikes/s"); plt.show()

**Result: reproduced (not an independent test).** At the midpoints the average maps give 26.48 (S40), 19.14 (S52), 8.22 (S90) and 3.79 spikes/s (P180), ordered S40 > S52 > S90 > P180, and the mean over the ten blocks gives the same order (23.04, 17.16, 7.44, 3.47). As the claims file notes, the midpoint reading repeats the means of fig4b-spike-rate-difference.

## Table 1: blocks along the folds and the Edge

Ten blocks of 10% luminance, 8 DU wide, from 0% to 100% luminance. Readings: (a) per neuron the
minimum, maximum, mean and median of its ten blocks, then the mean over neurons; (b) the same summaries
of the population-average profile. For S90 and S52: (i) the two halves averaged block by block; (ii)
each half its own set of ten blocks (summaries per half, then averaged).

In [ ]:
T1 = {"Edge": ("Edge-compare", [7.98, 33.13, 20.44, 20.77, 22.01], [1.60, 5.11, 3.33, 3.48, 3.41]),
      "P180": ("Edge-compare", [0.18, 10.28, 4.55, 4.03, 4.87], [1.01, 1.53, 1.12, 1.06, 1.15]),
      "S90": ("Edge-compare", [4.36, 14.61, 9.22, 9.31, 10.69], [1.07, 2.48, 1.61, 1.62, 1.94]),
      "S52": ("Star", [6.26, 25.39, 15.66, 15.93, 18.10], [1.33, 3.52, 2.23, 2.36, 2.39]),
      "S40": ("Corners", [9.33, 35.11, 22.81, 22.83, 25.60], [1.69, 4.17, 3.11, 3.48, 4.18])}
STATS = ["min", "max", "mean", "median", "midpoint"]
per.loc[per.folder == "Edge-compare", "diff_Edge"] = per.midpoint_Edge - per.base_main

def block_arrays(fold, f):
    d = per[per.folder == f]
    B = np.stack(d[f"blocks_{fold}"].values) - d.base_main.values[:, None, None]   # neurons x halves x 10
    return B, d[f"diff_{fold}"].values

def table1(fold):
    f, exp, tol = T1[fold]
    B, mid = block_arrays(fold, f)
    out = {}
    def summ(X):   # X: units x 10
        return [np.min(X, 1), np.max(X, 1), np.mean(X, 1), np.median(X, 1)]
    variants = {"(i)": B.mean(1)}
    if B.shape[1] == 2:
        variants["(ii)"] = None
    for half, X in variants.items():
        if half == "(ii)":
            sa = [np.mean([s.mean() for s in summ(B[:, h])], axis=0) for h in range(2)]
            per_half = [summ(B[:, h]) for h in range(2)]
            a_vals = [np.mean([per_half[h][k].mean() for h in range(2)]) for k in range(4)]
            prof = [B[:, h].mean(0) for h in range(2)]
            b_vals = [np.mean([fn(p) for p in prof]) for fn in (np.min, np.max, np.mean, np.median)]
        else:
            a_vals = [s.mean() for s in summ(X)]
            prof = X.mean(0)
            b_vals = [prof.min(), prof.max(), prof.mean(), np.median(prof)]
        sem = [s.std(ddof=1) / np.sqrt(len(s)) for s in summ(B.mean(1))] + [mid.std(ddof=1) / np.sqrt(len(mid))]
        for rd, valsr in [("(a)", a_vals), ("(b)", b_vals)]:
            vv = list(valsr) + [mid.mean()]
            within = [abs(x - e) <= t for x, e, t in zip(vv, exp, tol)]
            out[f"{rd}{half if B.shape[1] == 2 else ''}"] = (vv, within)
    rows = []
    for k, (vv, within) in out.items():
        rows.append(dict(reading=k, **{s: round(x, 2) for s, x in zip(STATS, vv)}, n_within=int(sum(within))))
    rows.append(dict(reading="paper", **dict(zip(STATS, exp)), n_within=None))
    rows.append(dict(reading="tolerance", **dict(zip(STATS, tol)), n_within=None))
    print(f"{fold}: n = {B.shape[0]} neurons, {B.shape[1]} fold line(s); block length %.2f DU; SEM of reading (a)(i):"
          % (abs(v.geometry(recs[(f, per[per.folder == f].code.iloc[0])])[fold][0][2] - v.geometry(recs[(f, per[per.folder == f].code.iloc[0])])[fold][0][1]) / 10),
          np.round(sem, 2))
    print(pd.DataFrame(rows).to_string(index=False))
    print("profile (mean over neurons, 0% to 100%):", np.round(B.mean((0, 1)), 2))
    return out
T1_OUT = {}

### table1-edge

> Along the Edge, the spike rate difference over the ten 10% luminance blocks has a mean minimum of 7.98, maximum 33.13, mean 20.44 and median 20.77 spikes/s, and 22.01 at the Edge midpoint.

In [ ]:
T1_OUT["Edge"] = table1("Edge")
# Placement of the Edge blocks across the Edge line, which the paper does not state: centred on the
# line (main, as the RMS square is "centered on the Edge"), or 8 DU wide just inside the rectangle,
# or just outside it (in the 0% background), for comparison.
exp_e, tol_e = T1["Edge"][1], T1["Edge"][2]
de = per[per.folder == "Edge-compare"]
for label, shift in [("centred on the Edge line (main)", 0), ("inside the rectangle", 4), ("outside the rectangle", -4)]:
    Bs, mids = [], []
    for r in de.itertuples():
        rec = recs[("Edge-compare", r.code)]
        x, y0, y1 = v.geometry(rec)["Edge"][0]
        Bs.append(v.blocks(rec["smoothed"], (x + shift, y0, y1)) - r.base_main)
        mids.append(v.midpoint(rec["smoothed"], [(x + shift, y0, y1)]) - r.base_main)
    Bs, mids = np.array(Bs), np.array(mids)
    vals_e = [Bs.min(1).mean(), Bs.max(1).mean(), Bs.mean(1).mean(), np.median(Bs, 1).mean(), mids.mean()]
    print(f"{label:32s} (a):", np.round(vals_e, 2), "within:", int(sum(abs(a - e) <= t for a, e, t in zip(vals_e, exp_e, tol_e))))

**Result: not reproduced.** Under reading (a) the Edge blocks give minimum 5.51, maximum 29.86, mean 16.92 and median 16.78 spikes/s and the midpoint 18.72, so 2 of 5 values are within tolerance (maximum and midpoint). Reading (b) also gives 2 of 5. Values are lower than the paper's: the mean is 3.52 below 20.44 and the median 3.99 below 20.77. Placing the blocks inside the rectangle or outside it gives 2 and 1 of 5.

### table1-p180

> Along P180, the spike rate difference over the ten blocks has a mean minimum of 0.18, maximum 10.28, mean 4.55 and median 4.03 spikes/s, and 4.87 at the midpoint.

In [ ]:
T1_OUT["P180"] = table1("P180")

**Result: partially reproduced.** Under reading (a), P180 gives minimum -1.71, maximum 10.28, mean 3.47, median 2.84 and midpoint 3.88 spikes/s. Maximum, mean and midpoint are within tolerance, 3 of 5. The minimum is 1.89 below 0.18 and the median 1.19 below 4.03. Reading (b) also gives 3 of 5.

### table1-s90

> Along the S90 fold of the Square, the spike rate difference over the ten blocks has a mean minimum of 4.36, maximum 14.61, mean 9.22 and median 9.31 spikes/s, and 10.69 at the midpoint.

In [ ]:
T1_OUT["S90"] = table1("S90")

**Result: not reproduced.** Every S90 summary is below the paper's, and only the maximum is within tolerance under readings (a)(i) and (a)(ii): 13.43 and 15.36 against 14.61 ± 2.48, so 1 of 5. Minima are 2.30 and 1.30 (paper 4.36), means 7.46 (9.22), medians 7.50 and 7.23 (9.31), and the midpoint 8.19 (10.69). Readings (b)(i) and (b)(ii) give 0 of 5.

### table1-s52

> Along the S52 fold of the Star, the spike rate difference over the ten blocks has a mean minimum of 6.26, maximum 25.39, mean 15.66 and median 15.93 spikes/s, and 18.10 at the midpoint.

In [ ]:
T1_OUT["S52"] = table1("S52")

**Result: reproduced (on 12 of 30 Star neurons).** Under reading (a)(ii), each half of the aligned diagonal taken as its own ten blocks, all five values are within tolerance: minimum 7.46, maximum 28.71, mean 17.11, median 16.43 and midpoint 19.01 spikes/s (paper 6.26, 25.39, 15.66, 15.93, 18.10). Reading (a)(i) gives 4 of 5 (minimum 9.03 is outside), and (b)(i) and (b)(ii) give 3 of 5.

### table1-s40

> Along the S40 fold of Corners, the spike rate difference over the ten blocks has a mean minimum of 9.33, maximum 35.11, mean 22.81 and median 22.83 spikes/s, and 25.60 at the midpoint.

In [ ]:
T1_OUT["S40"] = table1("S40")

**Result: reproduced.** Under reading (a), all five S40 values are within tolerance: minimum 10.22, maximum 34.36, mean 23.04, median 23.06 and midpoint 26.48 spikes/s (paper 9.33, 35.11, 22.81, 22.83, 25.60). Reading (b) gives 3 of 5.

## fig6-midpoint-representative

> For each fold and the Edge, the spike rate difference at the midpoint does not differ significantly
> from the mean or the median spike rate difference along the fold or Edge, so the midpoint response
> represents the whole fold.

Wilcoxon rank sum tests of per-neuron midpoints against per-neuron block means and medians (blocks as
reading (a)(i) of Table 1), alpha 0.05, no correction; signed rank test as a variant.

In [ ]:
rows = []
for fold in ["Edge", "P180", "S90", "S52", "S40"]:
    f = T1[fold][0]
    B, mid = block_arrays(fold, f)
    X = B.mean(1)
    for name, s in [("mean", X.mean(1)), ("median", np.median(X, 1))]:
        rows.append(dict(fold=fold, n=len(mid), against=name, median_midpoint=np.median(mid), median_other=np.median(s),
                         median_paired_difference=np.median(mid - s), p_rank_sum=stats.ranksums(mid, s).pvalue,
                         p_signed_rank=stats.wilcoxon(mid, s).pvalue))
f6 = pd.DataFrame(rows)
print(f6.round(4).to_string(index=False))
print("rank sum tests with p < 0.05:", int((f6.p_rank_sum < 0.05).sum()), "of", len(f6),
      "| signed rank tests with p < 0.05:", int((f6.p_signed_rank < 0.05).sum()))

**Result: reproduced.** None of the ten rank sum tests is significant; the smallest p is 0.43 (S40 midpoint against block medians). Paired median differences (midpoint minus block summary) range from -0.20 to 2.17 spikes/s. With the signed rank test, which pairs the values, 3 of 10 are significant: S90 against medians (p = 0.025), and S40 against means (0.013) and medians (0.0086).

## fig7a-rms-contrast-fit

> Local RMS contrast along the Edge rises from 0.61 to 0.943 with luminance and is fit by an
> exponential model, f(l) = -0.34 exp(-0.05 l) + 0.95, with R2 = 0.9983.

The stimulus luminance in the aligned frame is rasterised from the polygons (the Edge geometry is the
same in all 22 files). RMS contrast in a 16 x 16 DU square centred on the Edge from 0% to 100% in 1.5%
steps; readings (a) SD over mean, (b) SD of luminance scaled to 0-1. Fit f(l) = a exp(-b l) + c by
Levenberg-Marquardt, then least absolute residuals from that start.

In [ ]:
edge_rec = recs[("Edge-compare", per[per.folder == "Edge-compare"].code.iloc[0])]
levels = np.arange(0, 100.0001, 1.5)
prof = v.rms_profile(edge_rec, levels)
def expo(l, a, b, c):
    return a * np.exp(-b * l) + c
fits = {}
for col in ["rms_a", "rms_b"]:
    y = prof[col].values; ok = np.isfinite(y); l = levels[ok]; y = y[ok]
    try:
        p0, _ = curve_fit(expo, l, y, p0=[-0.34, 0.05, 0.95], method="lm", maxfev=20000)
        lar = minimize(lambda q: np.abs(expo(l, *q) - y).sum(), p0, method="Nelder-Mead",
                       options=dict(maxiter=20000, xatol=1e-8, fatol=1e-10)).x
        r2 = 1 - np.sum((y - expo(l, *lar)) ** 2) / np.sum((y - y.mean()) ** 2)
    except RuntimeError as err:
        lar, r2 = [np.nan] * 3, np.nan
    fits[col] = dict(lowest=y.min(), highest=y.max(), a=lar[0], b=lar[1], c=lar[2], R2=r2, n_levels=len(y))
exp7a = dict(lowest=(0.61, 0.061), highest=(0.943, 0.094), a=(-0.34, 0.034), b=(0.05, 0.005), c=(0.95, 0.095), R2=(0.9983, 0.0017))
t = pd.DataFrame(fits).T
for k, (e, tol) in exp7a.items():
    t[f"{k}_ok"] = (t[k] - e).abs() <= tol
print(t.round(4).to_string())
print("paper:", {k: e for k, (e, _) in exp7a.items()})
print(prof.iloc[::6].round(3).to_string(index=False))

**Result: not reproduced.** Neither definition of RMS contrast in the claims file gives the paper's curve on this stimulus. SD over mean runs from 1.05 to 2.04, is near 1.06 to 1.13 over most of the Edge, and fits with R2 = 0.73. SD of luminance scaled to 0 to 1 rises from 0.016 to 0.468, almost linearly (fit a = -149.6, R2 = 0.996). No fitted value is within tolerance under either reading. The paper's values (0.61 to 0.943) imply a definition not given in the text read (reference 21).

## fig7c-fold-edge-equivalence

> Neural responses to P180 are roughly equivalent to those to a 50% contrast Edge, while responses to
> S40 are close to those to a 90% contrast Edge, so sharper corners produce responses equivalent to
> higher contrasts along a straight edge.

Edge responses: spike rate difference in a 16 x 16 DU square centred on the Edge. Reading (a): Edge
luminance level, at 10% intervals; (a1) the ten Table 1 blocks (centres 5% to 95%), (a2) 16 x 16
squares at 0%, 10%, ..., 100%. Reading (b): RMS contrast x 100 at the ten levels 0.610 to 0.943, mapped
to luminance by the inverse of the paper's fit. Refit variant: not run, as neither RMS reading fits the
paper's curve (fig7a).

In [ ]:
A, B_, C = -0.34, 0.05, 0.95
rms_levels = 0.610 + 0.037 * np.arange(10)
lum_for_rms = -np.log((rms_levels - C) / A) / B_
print("luminance (%) at the ten RMS levels:", np.round(lum_for_rms, 1))
de = per[per.folder == "Edge-compare"]
edge_rms = np.array([[v.edge_window(recs[("Edge-compare", c)], l) for l in lum_for_rms] for c in de.code]) - de.base_main.values[:, None]
edge_sq = np.array([[v.edge_window(recs[("Edge-compare", c)], l) for l in np.arange(0, 101, 10)] for c in de.code]) - de.base_main.values[:, None]
edge_blk = np.stack(de.blocks_Edge.values)[:, 0, :] - de.base_main.values[:, None]
fold_mean = {fold: per.loc[per.folder == f, f"diff_{fold}"].mean() for fold, f in FOLDS4}
print("fold means:", {k: round(x, 2) for k, x in fold_mean.items()})
def nearest(levels, resp, target):
    return levels[int(np.argmin(np.abs(resp - target)))]
rows = []
for name, lev, resp, step, t50, t90 in [
        ("(a1) blocks, luminance %", np.arange(5, 100, 10.0), edge_blk.mean(0), 10, 50, 90),
        ("(a2) 16x16 squares, luminance %", np.arange(0, 101, 10.0), edge_sq.mean(0), 10, 50, 90),
        ("(b) RMS x 100", rms_levels * 100, edge_rms.mean(0), 3.7, None, 90)]:
    n180, n40 = nearest(lev, resp, fold_mean["P180"]), nearest(lev, resp, fold_mean["S40"])
    ok180 = (abs(n180 - 50) <= step + 1e-9) if t50 else bool(np.isclose(n180, lev.min()))
    ok40 = abs(n40 - 90) <= step + 1e-9
    rows.append(dict(reading=name, edge_responses=np.round(resp, 1).tolist(), P180_nearest=round(n180, 1),
                     P180_ok=ok180, S40_nearest=round(n40, 1), S40_ok=ok40))
    for fold in ["S90", "S52"]:
        rows[-1][f"{fold}_nearest"] = round(nearest(lev, resp, fold_mean[fold]), 1)
f7c = pd.DataFrame(rows)
print(f7c.to_string(index=False))
print("Edge response range over the RMS levels %.2f to %.2f spikes/s; SEM per level:" % (edge_rms.mean(0).min(), edge_rms.mean(0).max()),
      np.round(edge_rms.std(0, ddof=1) / np.sqrt(len(edge_rms)), 2))

**Result: not reproduced.** No reading meets both conditions. The P180 mean (3.88 spikes/s) is below every Edge response and the S40 mean (26.48) above every one: the Edge ranges from 8.61 to 20.04 spikes/s over the RMS levels. Reading (b) meets the P180 condition only through the rule for the lowest level (61.0), and the S40 nearest is 94.3, 4.3 from 90 where one step is 3.7. Reading (a2) puts S40 nearest 90% but P180 nearest 0%. Reading (a1) puts them at 5% and 75%.

## fig7c-fold-edge-significance

> Responses to P180 are significantly lower than responses to a 94% contrast Edge, and responses to
> S40 are significantly higher than responses to an Edge of less than 68% contrast.

Two-sample t-tests of per-neuron fold midpoint differences against per-neuron Edge responses at each of
the ten RMS levels. Corrections: (a) BH over 40 tests; (b) BH over the ten levels per fold; (c)
Bonferroni over 40 and over 10.

In [ ]:
rows = []
for fold, f in FOLDS4:
    x = per.loc[per.folder == f, f"diff_{fold}"].values
    for k, lev in enumerate(rms_levels):
        res = stats.ttest_ind(x, edge_rms[:, k])
        rows.append(dict(fold=fold, rms=round(lev, 3), mean_difference=x.mean() - edge_rms[:, k].mean(), t=res.statistic, p=res.pvalue))
tt = pd.DataFrame(rows)
tt["bh40"] = multipletests(tt.p, method="fdr_bh")[1]
tt["bh10"] = tt.groupby("fold").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
tt["bonf40"] = np.minimum(tt.p * 40, 1)
tt["bonf10"] = np.minimum(tt.p * 10, 1)
print(tt[tt.fold.isin(["P180", "S40"])].round(4).to_string(index=False))
for corr in ["p", "bh40", "bh10", "bonf40", "bonf10"]:
    a = tt[(tt.fold == "P180") & np.isclose(tt.rms, 0.943)]
    b = tt[(tt.fold == "S40") & tt.rms.isin([0.61, 0.647])]
    p180 = bool(((a[corr] < 0.05) & (a.mean_difference < 0)).all())
    s40 = bool(((b[corr] < 0.05) & (b.mean_difference > 0)).all())
    sig40 = tt[(tt.fold == "S40") & (tt[corr] < 0.05) & (tt.mean_difference > 0)].rms.tolist()
    sig180 = tt[(tt.fold == "P180") & (tt[corr] < 0.05) & (tt.mean_difference < 0)].rms.tolist()
    print(f"{corr:7s} P180 < Edge at 0.943: {p180}; S40 > Edge at 0.610 and 0.647: {s40}; "
          f"S40 higher at {sig40}; P180 lower at {sig180}")

**Result: reproduced.** P180 is lower than the Edge at RMS 0.943 (mean difference -16.16 spikes/s, p = 0.0005), and S40 higher than the Edge at 0.610 (17.87, p = 0.0007) and 0.647 (17.41, p = 0.0009). Both hold under every correction: BH over 40 tests, BH over 10 per fold, Bonferroni over 40 and over 10. With BH over 40, S40 is also higher at 0.684 (p = 0.011) and up to 0.795, and P180 is lower at all ten levels; with Bonferroni over 40, only at the two lowest and two highest levels, respectively.

## fig7d-nonlinear

> The relationship between corner angle and V1 spike rate difference is parametric but apparently not
> linear; the paper says it cannot be certain of this, given the few angles tested.

Reading: least-squares line through the four means against angle; non-linear when at least one mean
lies off the line by more than its SEM. Variant: slopes between successive angles compared with their
propagated SEMs.

In [ ]:
ang = np.array([ANGLE[fold] for fold, _ in FOLDS4], float)
mu = np.array([per.loc[per.folder == f, f"diff_{fold}"].mean() for fold, f in FOLDS4])
se = np.array([per.loc[per.folder == f, f"diff_{fold}"].sem() for fold, f in FOLDS4])
slope, icpt = np.polyfit(ang, mu, 1)
resid = mu - (slope * ang + icpt)
print(pd.DataFrame(dict(angle=ang, mean=mu, sem=se, residual=resid, off_by_more_than_sem=np.abs(resid) > se)).round(2).to_string(index=False))
print("line: %.3f spikes/s per degree, intercept %.2f; means decrease monotonically with angle: %s"
      % (slope, icpt, bool(np.all(np.diff(mu[np.argsort(ang)]) < 0))))
order = np.argsort(ang)
a_s, m_s, s_s = ang[order], mu[order], se[order]
sl = np.diff(m_s) / np.diff(a_s)
sl_se = np.sqrt(s_s[1:] ** 2 + s_s[:-1] ** 2) / np.diff(a_s)
print("variant, slopes 40-52, 52-90, 90-180 (spikes/s per degree):", np.round(sl, 3), "SE", np.round(sl_se, 3))
for i in range(len(sl)):
    for j in range(i + 1, len(sl)):
        print(f"  slope {i} vs {j}: difference %.3f, propagated SE %.3f, differ by more: %s"
              % (sl[i] - sl[j], np.hypot(sl_se[i], sl_se[j]), abs(sl[i] - sl[j]) > np.hypot(sl_se[i], sl_se[j])))

**Result: reproduced.** The means decrease monotonically with angle (26.48, 19.01, 8.19, 3.88 spikes/s at 40, 52, 90 and 180 degrees). A straight line leaves three of the four means off by more than their SEM: residuals 4.85 at 40, -6.27 at 90 and 2.32 at 180 degrees, against SEMs of 4.28, 1.98 and 0.91. In the variant, the slope from 90 to 180 degrees (-0.048 per degree) differs from those from 40 to 52 (-0.622) and from 52 to 90 (-0.285) by more than the propagated SE, and the first two do not differ from each other. S52 rests on 12 of 30 Star neurons.

## fig7d-dog-similar and s3-latency-peak

Blocked. The DOG model output of reference [3] is a published figure, with no data in this dataset or
any public archive found. The latency analysis needs spike times and eye positions; the dataset holds
maps made at the paper's 60 ms latency only.

## Summary

| Claim | Outcome |
|---|---|
| neuron-counts | not reproduced |
| spikes-per-condition | blocked, data missing |
| s4-baseline-no-difference | not reproduced (assumed baseline placement) |
| fig4a-above-baseline | reproduced |
| fig4b-spike-rate-difference | partially reproduced |
| fig4b-pairwise-significance | reproduced |
| fig5-average-maps | reproduced |
| table1-edge | not reproduced |
| table1-p180 | partially reproduced |
| table1-s90 | not reproduced |
| table1-s52 | reproduced |
| table1-s40 | reproduced |
| fig6-midpoint-representative | reproduced |
| fig7a-rms-contrast-fit | not reproduced |
| fig7c-fold-edge-equivalence | not reproduced |
| fig7c-fold-edge-significance | reproduced |
| fig7d-nonlinear | reproduced |
| fig7d-dog-similar | blocked, data missing |
| s3-latency-peak | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: it starts from the authors' aligned spike rate maps, so the eye-signal correction, the 60 ms registration and the smoothing are theirs, not repeated. The data set holds 53 of the 58 neurons. Star rests on the 12 of 27 Star neurons whose files carry the aligned frame, against 30 in the paper. The Corners baseline squares are placed at the bottom corners of the frame, an assumption, as Fig 2C is not in the text. The Square and Edge and Star squares sit inside the corners of the circumscribing square, and the stored `BaseActivity` differs from both placements.

What was found about the dataset:
- It holds 53 neurons, against 58 analysed in the paper. Its sheet lists 58 cells, 5 of them with no files.
- `image` is spikes per eye sample (spikes/ms), not raw spike counts as the description says. Eye time is stored in 1 of 75 files, so spike counts cannot be recovered.
- 2 files in the Star folder (ABS042700J2, ABS051000J3) hold a Corners-type stimulus, a single notched corner, and no aligned frame. One of them is in MATLAB 7.3 format.
- 15 of the 27 Star files have no aligned frame. The screen-frame maps are in gaze coordinates, and the receptive-field offset and the aligned diagonal are not stored, so these neurons cannot be registered to the stimulus.
- The aligned frame, the stored `BaseActivity`, `smoothed_image_norm` and `eyetime` are not described. `BaseActivity` is present in 38 of 60 aligned files and is 0 for two Corners neurons. `smoothed_image_norm` equals (map - BaseActivity)/(max - BaseActivity) in 21 of 38 files.
- The Corners frames come in two sizes (273 x 209 and 257 x 193).

In [ ]:
print("total runtime, s: %.0f; peak memory, MB: %.0f" % (time.time() - T0, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024))